## Kaggle: restore results_file1.zip
Kaggle auto-unzips uploaded archives on ingest. This cell re-zips the extracted File 1 output back into `/kaggle/working/results_file1.zip` so Cell 1's existing logic (unchanged below) finds it as expected. Run this before `## 0. Setup`.

In [1]:
import os, subprocess

# 1) Does /kaggle/temp exist / is it its own mount (vs just a folder on the same
#    small disk as /kaggle/working)?
print("=== mount check ===")
subprocess.run("df -h /kaggle/working /kaggle/temp 2>&1 || true", shell=True)

# 2) If /kaggle/temp doesn't exist yet, create it and check again
os.makedirs("/kaggle/temp", exist_ok=True)
print("\n=== after mkdir ===")
subprocess.run("df -h /kaggle/temp", shell=True)

# 3) Sanity write test - make sure it's actually writable and see free space in GB
import shutil
total, used, free = shutil.disk_usage("/kaggle/temp")
print(f"\n/kaggle/temp free space: {free/1e9:.1f} GB")

=== mount check ===
df: /kaggle/temp: No such file or directory
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  8.0G  60% /kaggle/working

=== after mkdir ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         7.9T  6.9T  1.1T  88% /

/kaggle/temp free space: 1102.5 GB


In [2]:
import zipfile, glob
from pathlib import Path

# Kaggle auto-unzips uploaded .zip files, so results_file1.zip arrives under
# /kaggle/input as an already-extracted folder. Find it, then re-zip its
# contents (flat, no wrapper folder) into /kaggle/working/results_file1.zip
# so Cell 1 below can find and extract it exactly as it expects.
marker = glob.glob("/kaggle/input/**/acoustic_profile.json", recursive=True)

if marker:
    src_dir = Path(marker[0]).parent
    out_zip = Path("/kaggle/working/results_file1.zip")
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir))
    print(f"Re-zipped File 1 results ({src_dir}) -> {out_zip} "
          f"({out_zip.stat().st_size/1e6:.2f} MB)")
else:
    zsrc = glob.glob("/kaggle/input/**/results_file1.zip", recursive=True)
    if zsrc:
        import shutil; shutil.copy(zsrc[0], "/kaggle/working/results_file1.zip")
        print("results_file1.zip was not auto-extracted; copied directly:", zsrc[0])
    else:
        print("WARNING: could not find File 1 results under /kaggle/input. "
              "Check that you added it via + Add Input.")


Re-zipped File 1 results (/kaggle/input/datasets/moitrayan/codec-checkpoints) -> /kaggle/working/results_file1.zip (0.02 MB)


# File 2 — GPU Training (Kaggle, PyTorch)

**What this does:** ResNet-50 + CNN training — sweep across 8 AMR-NB modes,
clean/matched/augmented regimes, embeddings, representation shift.
**Input needed:** upload `results_file1.zip` from File 1 (add it via *Add Data* as a Kaggle Dataset, or upload directly to `/kaggle/working/`).
**Output:** `results_file2.zip` — download when done.
**Runtime:** ~8–12 hours on T4. Enable GPU: Notebook Settings (right panel) → Accelerator → GPU T4 x2 (or P100).
**Checkpointed:** every fold saved; re-run after timeout to resume.

## 0. Setup

In [3]:
import os, sys, json, csv, random, subprocess
from pathlib import Path
import numpy as np
SEED=42; random.seed(SEED); np.random.seed(SEED)

# /kaggle/temp = large scratch space, NOT saved with notebook output (wiped between
# sessions). Used below only for re-downloadable / re-derivable process artifacts
# (raw datasets, degraded-audio cache). Nothing that helps resume a killed session
# lives there.
TEMP=Path("/kaggle/working/temp"); TEMP.mkdir(parents=True,exist_ok=True)

# unzip File 1 results if present
f1=Path("results_file1.zip")
if f1.exists():
    import zipfile
    with zipfile.ZipFile(f1) as z: z.extractall(TEMP/"results_f1")
    print("File 1 results loaded.")
else:
    print("WARNING: results_file1.zip not found. Upload it to the Files panel.")

RESULTS=Path("results"); RESULTS.mkdir(exist_ok=True)
FIGS=RESULTS/"figs"; FIGS.mkdir(exist_ok=True)
CKPT=RESULTS/"ckpt"; CKPT.mkdir(exist_ok=True)
DEGRADED=TEMP/"degraded"; DEGRADED.mkdir(exist_ok=True)

# copy acoustic files from File 1 if available
for fname in ["acoustic_profile.json","hf_fraction_per_class.json",
              "mismatch_env_vs_speech.json"]:
    src=TEMP/"results_f1"/fname
    if src.exists():
        import shutil; shutil.copy(src,RESULTS/fname)
        print(f"  copied {fname} from File 1")

AMR_MODES=[4.75,5.15,5.90,6.70,7.40,7.95,10.20,12.20]; KEY_MODES=[4.75,12.20]
TARGET_SR=22050; MAX_DUR=4.0; N_MELS=128; N_FFT=2048; HOP=512
BANDS=[(0,500),(500,1000),(1000,2000),(2000,3400),(3400,4000),(4000,8000),(8000,11025)]
EQUIV_MARGIN=0.05; BATCH=32; EPOCHS=30; PATIENCE=5
LR_RESNET=1e-4; LR_CNN=1e-3
RUN_SWEEP=True; RUN_REGIMES=True; RUN_EMBED=True

import torch
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",DEVICE)
if DEVICE.type=="cpu": print("WARNING: No GPU detected. Enable: Runtime → Change runtime type → T4 GPU")
torch.manual_seed(SEED)

File 1 results loaded.
  copied acoustic_profile.json from File 1
  copied hf_fraction_per_class.json from File 1
  copied mismatch_env_vs_speech.json from File 1
device: cuda


## 0b. Resume from checkpoints (if any)
If a Kaggle Dataset containing checkpoint files from a previous, timed-out session
has been attached to this notebook (**+ Add Input**), this copies every
`*_agg.json` / `*_f<N>.json` file it finds anywhere under `/kaggle/input` into the
local `ckpt/` folder in the working directory. The training loop below (unchanged)
already checks for these files before training each fold/mode, so anything found
here is skipped automatically and training resumes from the first missing piece.
Safe to run even with no such dataset attached — it just reports "starting fresh".


In [4]:
import glob as _cglob, shutil as _cshutil

# Find previously-completed checkpoint files anywhere under /kaggle/input,
# regardless of what the attached Kaggle Dataset happens to be named.
_ckpt_files = sorted(set(
    _cglob.glob("/kaggle/input/**/*_agg.json", recursive=True) +
    _cglob.glob("/kaggle/input/**/*_f[0-9]*.json", recursive=True)
))

_copied = 0
for _src in _ckpt_files:
    _dest = CKPT / Path(_src).name
    if not _dest.exists():
        _cshutil.copy(_src, _dest); _copied += 1

_existing = sorted(CKPT.glob("*.json"))
if _existing:
    print(f"Resume: found {len(_ckpt_files)} checkpoint file(s) under /kaggle/input, "
          f"copied {_copied} new one(s) into {CKPT}.")
    print(f"  {len(_existing)} checkpoint file(s) now available in the working dir "
          f"— matching runs below will be skipped.")
else:
    print("Resume: no prior checkpoint files found under /kaggle/input/** — starting fresh.")


Resume: found 24 checkpoint file(s) under /kaggle/input, copied 0 new one(s) into results/ckpt.
  320 checkpoint file(s) now available in the working dir — matching runs below will be skipped.


## 1. AMR ffmpeg + deps

In [5]:
subprocess.run("apt-get update -qq && apt-get install -y -qq libavcodec-extra "
               "libopencore-amrnb0 libopencore-amrwb0 ffmpeg",shell=True)
enc=subprocess.run(["ffmpeg","-encoders"],capture_output=True,text=True).stdout
if "libopencore_amrnb" not in enc:
    subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && "
        "wget -q https://johnvansickle.com/ffmpeg/releases/"
        "ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz",shell=True)
    d=[x for x in os.listdir("/kaggle/temp") if x.startswith("ffmpeg-") and x.endswith("-static")]
    if d: os.environ["PATH"]=os.path.abspath(os.path.join("/kaggle/temp",d[0]))+":"+os.environ["PATH"]
    enc=subprocess.run(["ffmpeg","-encoders"],capture_output=True,text=True).stdout
assert "libopencore_amrnb" in enc,"AMR encoder missing"
for p in ["librosa","soundfile","scikit-learn","statsmodels","umap-learn","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p])
print("deps OK.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package libaribb24-0:amd64.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../libaribb24-0_1.0.3-2_amd64.deb ...
Unpacking libaribb24-0:amd64 (1.0.3-2) ...
Selecting previously unselected package libopencore-amrnb0:amd64.
Preparing to unpack .../libopencore-amrnb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrnb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libopencore-amrwb0:amd64.
Preparing to unpack .../libopencore-amrwb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrwb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libvo-amrwbenc0:amd64.
Preparing to unpack .../libvo-amrwbenc0_0.1.3-2_amd64.deb ...
Unpacking libvo-amrwbenc0:amd64 (0.1.3-2) ...
dpkg: libavcodec58:amd64: dependency problems, but removing anyway as you requested:
 libchromaprint1:amd64 depends on libavcodec58 (>= 7:4.4).
 libavformat58:amd64 depends on libavcodec58 (= 7:4.4.2-0ubuntu0.22.04.1).
 libavfilter7:a

## 2. Datasets + degradation (skips cached)

In [6]:
import tarfile, zipfile, urllib.request, librosa, soundfile as sf
from tqdm.auto import tqdm
from multiprocessing.pool import ThreadPool; import multiprocessing as mp

def _dl_resume(url,dest,desc=""):
    if Path(dest).exists() and Path(dest).stat().st_size>1e6: return
    subprocess.run(f"wget -c --tries=5 --timeout=60 -O '{dest}' '{url}'",shell=True)

import glob as _glob

# UrbanSound8K was added directly as a Kaggle Input dataset (+ Add Input), so
# use that instead of downloading from Zenodo. ESC-50 and Speech Commands are
# not provided as Kaggle inputs, so they still download as before.
_us8k_csv = _glob.glob("/kaggle/input/**/UrbanSound8K.csv", recursive=True)
US8K = str(Path(_us8k_csv[0]).parent) if _us8k_csv else "/kaggle/temp/UrbanSound8K"
ESC="/kaggle/temp/ESC-50-master"; SC="/kaggle/temp/speech_commands"
Path("/kaggle/temp").mkdir(parents=True,exist_ok=True)
if _us8k_csv:
    print(f"US8K: using Kaggle input dataset at {US8K} (skipping Zenodo download)")
elif not Path(f"{US8K}/metadata/UrbanSound8K.csv").exists():
    _dl_resume("https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz?download=1",
               "/kaggle/temp/u.tgz","US8K")
    try:
        with tarfile.open("/kaggle/temp/u.tgz") as t: t.getmembers()
        with tarfile.open("/kaggle/temp/u.tgz") as t: t.extractall("/kaggle/temp")
        os.remove("/kaggle/temp/u.tgz")
    except Exception as e: print("US8K archive issue:",e,"— re-run cell")
if not Path(f"{ESC}/meta/esc50.csv").exists():
    urllib.request.urlretrieve("https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip","/kaggle/temp/e.zip")
    with zipfile.ZipFile("/kaggle/temp/e.zip") as z: z.extractall("/kaggle/temp"); os.remove("/kaggle/temp/e.zip")
if not Path(SC).exists() or not list(Path(f"{SC}/yes").glob("*.wav")):
    _dl_resume("http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz","/kaggle/temp/sc.tgz","SC")
    Path(SC).mkdir(parents=True,exist_ok=True)
    with tarfile.open("/kaggle/temp/sc.tgz") as t: t.extractall(SC); os.remove("/kaggle/temp/sc.tgz")

def us8k_rows():
    rows=[]
    # Kaggle input layout puts UrbanSound8K.csv at the dataset root with
    # fold1..fold10 directly under it; the Zenodo tar.gz extracts to
    # metadata/UrbanSound8K.csv with audio/fold1..fold10. Support both.
    csv_path = Path(US8K)/"UrbanSound8K.csv"
    if not csv_path.exists(): csv_path = Path(US8K)/"metadata"/"UrbanSound8K.csv"
    with open(csv_path) as fh:
        for r in csv.DictReader(fh):
            fold=int(r["fold"])
            p_flat = Path(US8K)/f"fold{fold}"/r["slice_file_name"]
            p_nested = Path(US8K)/"audio"/f"fold{fold}"/r["slice_file_name"]
            rows.append((str(p_flat) if p_flat.exists() else str(p_nested),r["class"],fold))
    return rows
def esc_rows():
    rows=[]
    with open(f"{ESC}/meta/esc50.csv") as fh:
        for r in csv.DictReader(fh): rows.append((f"{ESC}/audio/{r['filename']}",r["category"],int(r["fold"])))
    return rows
def sc_rows():
    KWS=["yes","no","up","down","left","right","on","off","stop","go"]; rows=[]; gi=0
    for kw in KWS:
        d=Path(SC)/kw
        if not d.exists(): continue
        for f in sorted(d.glob("*.wav"))[:200]: rows.append((str(f),kw,1+(gi%5))); gi+=1
    return rows
US8K_ROWS=us8k_rows(); ESC_ROWS=esc_rows(); SC_ROWS=sc_rows()
print(f"US8K={len(US8K_ROWS)} ESC50={len(ESC_ROWS)} SC={len(SC_ROWS)}")

def amr_nb(inp,out,kbps):
    stem=str(out)+".amr"
    r1=subprocess.run(["ffmpeg","-y","-i",str(inp),"-ac","1","-ar","8000","-b:a",f"{kbps}k",
        "-c:a","libopencore_amrnb","-f","amr",stem],capture_output=True,timeout=20)
    if r1.returncode!=0: raise RuntimeError(f"encode {Path(inp).name}")
    r2=subprocess.run(["ffmpeg","-y","-i",stem,"-ar",str(TARGET_SR),"-ac","1",str(out)],
        capture_output=True,timeout=20)
    if os.path.exists(stem): os.remove(stem)
    if r2.returncode!=0: raise RuntimeError(f"decode {Path(inp).name}")
def deg_path(name,k,p):
    base=DEGRADED/name
    for c in [f"amr{k}",f"amr{float(k):.2f}".rstrip("0").rstrip(".")]:
        d=base/c
        if d.exists(): return str(d/Path(p).name)
    return str(base/f"amr{k}"/Path(p).name)
def _deg_one(args):
    p,od,k=args; op=od/Path(p).name
    if op.exists(): return True
    try: amr_nb(p,op,k); return True
    except: return False
def degrade(rows,name,modes):
    n=max(1,mp.cpu_count()-1)
    for k in modes:
        od=DEGRADED/name/f"amr{k}"; od.mkdir(parents=True,exist_ok=True)
        todo=[(p,od,k) for p,_,_ in rows if not (od/Path(p).name).exists()]
        cached=len(rows)-len(todo)
        if not todo: print(f"  {name} AMR{k}: all {cached} cached"); continue
        with ThreadPool(n) as pool:
            list(tqdm(pool.imap(_deg_one,todo,chunksize=16),total=len(todo),desc=f"{name} AMR{k}",leave=False))
        print(f"  {name} AMR{k}: {len(list((DEGRADED/name/f'amr{k}').glob('*.wav')))} files")
degrade(US8K_ROWS,"us8k",AMR_MODES)
degrade(ESC_ROWS,"esc50",KEY_MODES); degrade(SC_ROWS,"sc",KEY_MODES)
print("degradation complete.")

US8K: using Kaggle input dataset at /kaggle/input/datasets/chrisfilo/urbansound8k (skipping Zenodo download)


--2026-08-30 17:18:03--  http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 64.233.181.207, 142.251.183.207, 209.85.200.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|64.233.181.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2428923189 (2.3G) [application/gzip]
Saving to: ‘/kaggle/temp/sc.tgz’

     0K .......... .......... .......... .......... ..........  0% 72.7M 32s
    50K .......... .......... .......... .......... ..........  0%  119M 26s
   100K .......... .......... .......... .......... ..........  0%  246M 20s
   150K .......... .......... .......... .......... ..........  0%  241M 18s
   200K .......... .......... .......... .......... ..........  0%  221M 16s
   250K .......... .......... .......... .......... ..........  0%  247M 15s
   300K .......... .......... .......... .......... ..........  0%  283M 14s
   350K .......... .......... ..

US8K=8732 ESC50=2000 SC=2000
  us8k AMR4.75: all 8732 cached
  us8k AMR5.15: all 8732 cached
  us8k AMR5.9: all 8732 cached
  us8k AMR6.7: all 8732 cached
  us8k AMR7.4: all 8732 cached
  us8k AMR7.95: all 8732 cached
  us8k AMR10.2: all 8732 cached
  us8k AMR12.2: all 8732 cached
  esc50 AMR4.75: all 2000 cached
  esc50 AMR12.2: all 2000 cached
  sc AMR4.75: all 2000 cached
  sc AMR12.2: all 2000 cached
degradation complete.


## 3. Models (PyTorch)

In [7]:
import torch, torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader

def to_mel(path):
    y,_=librosa.load(path,sr=TARGET_SR,mono=True,duration=MAX_DUR)
    L=int(TARGET_SR*MAX_DUR); y=np.pad(y,(0,max(0,L-len(y))))[:L]
    m=librosa.power_to_db(librosa.feature.melspectrogram(
        y=y,sr=TARGET_SR,n_mels=N_MELS,n_fft=N_FFT,hop_length=HOP),ref=np.max)
    return ((m-m.mean())/(m.std()+1e-6)).astype(np.float32)

class AudioDS(Dataset):
    def __init__(self,files,labels): self.files=files; self.labels=labels
    def __len__(self): return len(self.files)
    def __getitem__(self,i): return torch.tensor(to_mel(self.files[i])).unsqueeze(0),self.labels[i]

class ResNet50W(nn.Module):
    def __init__(self,nc):
        super().__init__(); self.chan=nn.Conv2d(1,3,1,bias=False)
        base=tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        for p in base.parameters(): p.requires_grad=False
        for p in base.layer4.parameters(): p.requires_grad=True
        self.backbone=nn.Sequential(*list(base.children())[:-1])
        self.penultimate=nn.Sequential(nn.Flatten(),nn.Linear(2048,128),nn.ReLU(),nn.Dropout(0.3))
        self.fc=nn.Linear(128,nc)
    def forward(self,x): return self.fc(self.penultimate(self.backbone(self.chan(x))))
    def get_embedding(self,x): return self.penultimate(self.backbone(self.chan(x)))

class LightCNN(nn.Module):
    def __init__(self,nc):
        super().__init__()
        layers=[]
        for ci,co in [(1,32),(32,64),(64,128)]:
            layers+=[nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(),nn.MaxPool2d(2)]
        self.conv=nn.Sequential(*layers)
        self.penultimate=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(128,128),nn.ReLU(),nn.Dropout(0.3))
        self.fc=nn.Linear(128,nc)
    def forward(self,x): return self.fc(self.penultimate(self.conv(x)))
    def get_embedding(self,x): return self.penultimate(self.conv(x))

MODELS={"resnet50":ResNet50W,"cnn":LightCNN}
x=torch.randn(2,1,128,173).to(DEVICE)
for name,cls in MODELS.items():
    m=cls(10).to(DEVICE); print(f"{name}: {m(x).shape} OK"); del m
torch.cuda.empty_cache(); print("models OK on",DEVICE)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s] 


resnet50: torch.Size([2, 10]) OK
cnn: torch.Size([2, 10]) OK
models OK on cuda


In [8]:
import hashlib

MELCACHE = Path("/kaggle/temp/melcache"); MELCACHE.mkdir(parents=True, exist_ok=True)

def _mel_cache_path(path):
    h = hashlib.md5(str(path).encode()).hexdigest()
    return MELCACHE/f"{h}.npy"

def to_mel_cached(path):
    cp = _mel_cache_path(path)
    if cp.exists():
        return np.load(cp)
    m = to_mel(path)          # original function from the Models cell, unchanged
    np.save(cp, m)
    return m

class AudioDS(Dataset):
    def __init__(self, files, labels): self.files=files; self.labels=labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return torch.tensor(to_mel_cached(self.files[i])).unsqueeze(0), self.labels[i]

print(f"Mel cache active at {MELCACHE}")

Mel cache active at /kaggle/temp/melcache


## 4. Training (checkpointed)

In [9]:
from sklearn.metrics import f1_score, confusion_matrix
import time
from tqdm.auto import tqdm

def fmt_time(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    if h: return f"{h}h {m}m {s}s"
    if m: return f"{m}m {s}s"
    return f"{s}s"

def load_split(rows,cond,folds,name):
    classes=sorted(set(c for _,c,_ in rows)); ci={c:i for i,c in enumerate(classes)}
    files=[]; labels=[]; skip=0
    for p,c,f in rows:
        if f not in folds: continue
        src=p if cond is None else deg_path(name,cond,p)
        if not os.path.exists(src): skip+=1; continue
        files.append(src); labels.append(ci[c])
    if not files: raise RuntimeError(f"no files (folds={folds},cond={cond},skip={skip})")
    return AudioDS(files,labels),classes

def train_fold(ds_tr,ds_te,arch,nc,lr,seed,tag):
    ck=CKPT/f"{tag}.json"
    if ck.exists():
        r=json.load(open(ck))
        print(f"  [{tag}] SKIPPED — already cached, F1={r['macro_f1']:.3f}")
        return r

    t_start = time.time()
    print(f"  [{tag}] STARTING — train={len(ds_tr)} samples, test={len(ds_te)} samples, "
          f"arch={arch}, lr={lr}, up to {EPOCHS} epochs (patience={PATIENCE})")

    torch.manual_seed(seed)
    model=MODELS[arch](nc).to(DEVICE)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)
    crit=nn.CrossEntropyLoss()
    val_n=max(1,len(ds_tr)//10); tr_n=len(ds_tr)-val_n
    ds_t,ds_v=torch.utils.data.random_split(ds_tr,[tr_n,val_n],generator=torch.Generator().manual_seed(seed))
    tl=DataLoader(ds_t,batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
    vl=DataLoader(ds_v,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)

    best=float("inf"); pat=0; bst=None
    epoch_bar = tqdm(range(EPOCHS), desc=f"    {tag} epochs", leave=False)
    for ep in epoch_bar:
        ep_start = time.time()
        model.train()
        train_loss = 0.0
        for X,y in tl:
            X,y=X.to(DEVICE),y.to(DEVICE); opt.zero_grad()
            loss = crit(model(X),y)
            loss.backward(); opt.step()
            train_loss += loss.item()
        train_loss /= len(tl)

        model.eval(); vl_=0.0
        with torch.no_grad():
            for X,y in vl: X,y=X.to(DEVICE),y.to(DEVICE); vl_+=crit(model(X),y).item()
        vl_/=len(vl)

        ep_time = time.time() - ep_start
        improved = vl_ < best
        if improved: best=vl_; pat=0; bst={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pat+=1

        epoch_bar.set_postfix({
            "train_loss": f"{train_loss:.4f}",
            "val_loss": f"{vl_:.4f}",
            "best": f"{best:.4f}",
            "patience": f"{pat}/{PATIENCE}",
            "sec/ep": f"{ep_time:.1f}",
        })
        print(f"    [{tag}] epoch {ep+1}/{EPOCHS}: train_loss={train_loss:.4f} "
              f"val_loss={vl_:.4f} {'(improved)' if improved else f'(no improve {pat}/{PATIENCE})'} "
              f"— {ep_time:.1f}s", flush=True)

        if pat>=PATIENCE:
            print(f"    [{tag}] early stopping at epoch {ep+1} (no improvement for {PATIENCE} epochs)")
            break

    model.load_state_dict(bst); model.eval()
    ap=[]; at=[]
    for X,y in DataLoader(ds_te,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True):
        ap+=model(X.to(DEVICE)).argmax(1).cpu().tolist(); at+=y.tolist()
    r=dict(macro_f1=float(f1_score(at,ap,average="macro",zero_division=0)),
           per_class=f1_score(at,ap,average=None,zero_division=0).tolist(),
           confusion=confusion_matrix(at,ap).tolist())
    json.dump(r,open(ck,"w")); del model,bst; torch.cuda.empty_cache()

    total_time = time.time() - t_start
    print(f"  [{tag}] DONE — F1={r['macro_f1']:.3f} — took {fmt_time(total_time)}")
    return r

def run_regime(name,rows,trc,tec,arch,seed,folds,tag):
    ck=CKPT/f"{tag}_agg.json"
    if ck.exists():
        print(f"[{tag}] SKIPPED — aggregate already cached")
        return json.load(open(ck))

    print(f"\n[{tag}] starting {len(folds)}-fold run (arch={arch})")
    t_start = time.time()
    classes=sorted(set(c for _,c,_ in rows)); nc=len(classes)
    lr=LR_RESNET if arch=="resnet50" else LR_CNN; per=[]
    fold_bar = tqdm(folds, desc=f"[{tag}] folds", leave=False)
    for i, te in enumerate(fold_bar):
        fold_bar.set_postfix({"fold": f"{te} ({i+1}/{len(folds)})"})
        tr=[f for f in folds if f!=te]
        if isinstance(trc,tuple) and trc[0]=="aug":
            dc,_=load_split(rows,None,tr,name); dd,_=load_split(rows,trc[1],tr,name)
            ds_tr=torch.utils.data.ConcatDataset([dc,dd])
        else: ds_tr,_=load_split(rows,trc,tr,name)
        ds_te,_=load_split(rows,tec,[te],name)
        per.append(train_fold(ds_tr,ds_te,arch,nc,lr,seed,f"{tag}_f{te}"))

        elapsed = time.time() - t_start
        avg_per_fold = elapsed / (i+1)
        remaining = avg_per_fold * (len(folds) - (i+1))
        print(f"  [{tag}] fold {te} complete — {i+1}/{len(folds)} folds done — "
              f"elapsed {fmt_time(elapsed)} — est. remaining for this regime: {fmt_time(remaining)}")

    r=dict(classes=classes,per_fold=per,f1all=[p["macro_f1"] for p in per],
           mean=float(np.mean([p["macro_f1"] for p in per])),
           std=float(np.std([p["macro_f1"] for p in per])),per_class_last=per)
    json.dump(r,open(ck,"w"))
    total_time = time.time() - t_start
    print(f"[{tag}] REGIME DONE — mean F1={r['mean']:.3f} (±{r['std']:.3f}) — took {fmt_time(total_time)}\n")
    return r

folds=list(range(1,11)); SEEDS=[42]
overall_start = time.time()

if RUN_SWEEP:
    sweep={}
    sweep_tasks = [(arch,k) for arch in ["resnet50","cnn"] for k in AMR_MODES]
    print(f"\n{'='*60}\nSWEEP: {len(sweep_tasks)} (arch, mode) combinations to process\n{'='*60}")
    for i, (arch, k) in enumerate(sweep_tasks):
        print(f"\n>>> SWEEP task {i+1}/{len(sweep_tasks)}: arch={arch}, AMR mode={k}")
        if arch not in sweep: sweep[arch]={}
        t0 = time.time()
        r=run_regime("us8k",US8K_ROWS,None,k,arch,SEED,folds,f"sweep_{arch}_{k}")
        sweep[arch][k]=dict(mean=r["mean"],std=r["std"],folds=r["f1all"])
        print(f"sweep {arch} AMR{k}: {r['mean']:.3f}  ({fmt_time(time.time()-t0)})")

        elapsed = time.time() - overall_start
        avg = elapsed / (i+1)
        remaining = avg * (len(sweep_tasks) - (i+1))
        print(f">>> SWEEP progress: {i+1}/{len(sweep_tasks)} tasks done — "
              f"elapsed {fmt_time(elapsed)} — est. remaining (sweep only): {fmt_time(remaining)}")
    json.dump(sweep,open(RESULTS/"bitrate_sweep.json","w"),indent=2)
    print(f"\nSWEEP COMPLETE — total time {fmt_time(time.time()-overall_start)}\n")

if RUN_REGIMES:
    regimes={}
    regime_start = time.time()
    regime_tasks = [(arch,k,label) for arch in ["resnet50","cnn"] for k in KEY_MODES
                     for label in [f"clean_to_clean", f"clean_to_{k}", f"matched_{k}", f"aug_{k}"]]
    print(f"\n{'='*60}\nREGIMES: {len(regime_tasks)} (arch, mode, regime) combinations to process\n{'='*60}")
    task_i = 0
    for arch in ["resnet50","cnn"]:
        regimes[arch]={}
        for k in KEY_MODES:
            specs={"clean_to_clean":(None,None),f"clean_to_{k}":(None,k),
                   f"matched_{k}":(k,k),f"aug_{k}":(("aug",k),k)}
            for label,(trc,tec) in specs.items():
                task_i += 1
                print(f"\n>>> REGIME task {task_i}/{len(regime_tasks)}: arch={arch}, mode={k}, regime={label}")
                t0 = time.time()
                f1all=[]; last=None
                for sd in SEEDS:
                    r=run_regime("us8k",US8K_ROWS,trc,tec,arch,sd,folds,f"{arch}_{label}_s{sd}")
                    f1all+=r["f1all"]; last=r
                regimes[arch][label]=dict(mean=float(np.mean(f1all)),std=float(np.std(f1all)),
                    f1all=f1all,classes=last["classes"],per_class_last=last["per_fold"])
                print(f"{arch} {label}: {np.mean(f1all):.3f}  ({fmt_time(time.time()-t0)})")

                elapsed = time.time() - regime_start
                avg = elapsed / task_i
                remaining = avg * (len(regime_tasks) - task_i)
                print(f">>> REGIME progress: {task_i}/{len(regime_tasks)} tasks done — "
                      f"elapsed {fmt_time(elapsed)} — est. remaining (regimes only): {fmt_time(remaining)}")
    json.dump(regimes,open(RESULTS/"regimes.json","w"),indent=2)
    print(f"\nREGIMES COMPLETE — total time {fmt_time(time.time()-regime_start)}\n")

print(f"{'='*60}\ntraining complete. total cell time: {fmt_time(time.time()-overall_start)}\n{'='*60}")


SWEEP: 16 (arch, mode) combinations to process

>>> SWEEP task 1/16: arch=resnet50, AMR mode=4.75
[sweep_resnet50_4.75] SKIPPED — aggregate already cached
sweep resnet50 AMR4.75: 0.371  (0s)
>>> SWEEP progress: 1/16 tasks done — elapsed 0s — est. remaining (sweep only): 0s

>>> SWEEP task 2/16: arch=resnet50, AMR mode=5.15
[sweep_resnet50_5.15] SKIPPED — aggregate already cached
sweep resnet50 AMR5.15: 0.368  (0s)
>>> SWEEP progress: 2/16 tasks done — elapsed 0s — est. remaining (sweep only): 0s

>>> SWEEP task 3/16: arch=resnet50, AMR mode=5.9
[sweep_resnet50_5.9] SKIPPED — aggregate already cached
sweep resnet50 AMR5.9: 0.391  (0s)
>>> SWEEP progress: 3/16 tasks done — elapsed 0s — est. remaining (sweep only): 0s

>>> SWEEP task 4/16: arch=resnet50, AMR mode=6.7
[sweep_resnet50_6.7] SKIPPED — aggregate already cached
sweep resnet50 AMR6.7: 0.442  (0s)
>>> SWEEP progress: 4/16 tasks done — elapsed 0s — est. remaining (sweep only): 0s

>>> SWEEP task 5/16: arch=resnet50, AMR mode=7.4


[cnn_aug_12.2_s42] folds:   0%|          | 0/10 [00:00<?, ?it/s]

  [cnn_aug_12.2_s42_f1] SKIPPED — already cached, F1=0.729
  [cnn_aug_12.2_s42] fold 1 complete — 1/10 folds done — elapsed 1s — est. remaining for this regime: 15s
  [cnn_aug_12.2_s42_f2] STARTING — train=15688 samples, test=888 samples, arch=cnn, lr=0.001, up to 30 epochs (patience=5)


    cnn_aug_12.2_s42_f2 epochs:   0%|          | 0/30 [00:00<?, ?it/s]

    [cnn_aug_12.2_s42_f2] epoch 1/30: train_loss=1.6263 val_loss=1.4806 (improved) — 456.9s
    [cnn_aug_12.2_s42_f2] epoch 2/30: train_loss=1.2463 val_loss=1.1913 (improved) — 13.5s
    [cnn_aug_12.2_s42_f2] epoch 3/30: train_loss=1.0763 val_loss=1.1819 (improved) — 13.5s
    [cnn_aug_12.2_s42_f2] epoch 4/30: train_loss=0.9583 val_loss=1.1436 (improved) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 5/30: train_loss=0.8639 val_loss=0.6713 (improved) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 6/30: train_loss=0.8108 val_loss=0.7985 (no improve 1/5) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 7/30: train_loss=0.7546 val_loss=0.8149 (no improve 2/5) — 13.7s
    [cnn_aug_12.2_s42_f2] epoch 8/30: train_loss=0.7082 val_loss=0.8766 (no improve 3/5) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 9/30: train_loss=0.6478 val_loss=0.6437 (improved) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 10/30: train_loss=0.6204 val_loss=0.6870 (no improve 1/5) — 13.6s
    [cnn_aug_12.2_s42_f2] epoch 11/30: train_loss=0.5987 val_los

...

    [cnn_aug_12.2_s42_f8] epoch 1/30: train_loss=1.5917 val_loss=2.0024 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 2/30: train_loss=1.1741 val_loss=1.6488 (improved) — 13.1s
    [cnn_aug_12.2_s42_f8] epoch 3/30: train_loss=1.0277 val_loss=1.0784 (improved) — 13.1s
    [cnn_aug_12.2_s42_f8] epoch 4/30: train_loss=0.9123 val_loss=1.2433 (no improve 1/5) — 13.1s
    [cnn_aug_12.2_s42_f8] epoch 5/30: train_loss=0.8569 val_loss=0.8962 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 6/30: train_loss=0.7746 val_loss=0.8420 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 7/30: train_loss=0.7331 val_loss=0.8142 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 8/30: train_loss=0.6905 val_loss=0.7120 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 9/30: train_loss=0.6492 val_loss=0.9335 (no improve 1/5) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 10/30: train_loss=0.6192 val_loss=0.6063 (improved) — 13.0s
    [cnn_aug_12.2_s42_f8] epoch 11/30: train_loss=0.5790 val_loss=0.7652 (no 

    cnn_aug_12.2_s42_f9 epochs:   0%|          | 0/30 [00:00<?, ?it/s]

    [cnn_aug_12.2_s42_f9] epoch 1/30: train_loss=1.5804 val_loss=1.6088 (improved) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 2/30: train_loss=1.1801 val_loss=1.0950 (improved) — 13.1s
    [cnn_aug_12.2_s42_f9] epoch 3/30: train_loss=1.0304 val_loss=1.0090 (improved) — 13.1s
    [cnn_aug_12.2_s42_f9] epoch 4/30: train_loss=0.9227 val_loss=0.9762 (improved) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 5/30: train_loss=0.8353 val_loss=0.9715 (improved) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 6/30: train_loss=0.7769 val_loss=1.3720 (no improve 1/5) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 7/30: train_loss=0.7165 val_loss=1.1485 (no improve 2/5) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 8/30: train_loss=0.6857 val_loss=0.8878 (improved) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 9/30: train_loss=0.6298 val_loss=0.6852 (improved) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 10/30: train_loss=0.6178 val_loss=0.7011 (no improve 1/5) — 13.0s
    [cnn_aug_12.2_s42_f9] epoch 11/30: train_loss=0.5803 val_loss=1.097

    cnn_aug_12.2_s42_f10 epochs:   0%|          | 0/30 [00:00<?, ?it/s]

    [cnn_aug_12.2_s42_f10] epoch 1/30: train_loss=1.5994 val_loss=1.5647 (improved) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 2/30: train_loss=1.2007 val_loss=1.1129 (improved) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 3/30: train_loss=1.0417 val_loss=0.9750 (improved) — 13.1s
    [cnn_aug_12.2_s42_f10] epoch 4/30: train_loss=0.9250 val_loss=0.8736 (improved) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 5/30: train_loss=0.8545 val_loss=1.2060 (no improve 1/5) — 12.9s
    [cnn_aug_12.2_s42_f10] epoch 6/30: train_loss=0.7934 val_loss=0.9576 (no improve 2/5) — 12.9s
    [cnn_aug_12.2_s42_f10] epoch 7/30: train_loss=0.7424 val_loss=0.5756 (improved) — 12.9s
    [cnn_aug_12.2_s42_f10] epoch 8/30: train_loss=0.6924 val_loss=1.5718 (no improve 1/5) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 9/30: train_loss=0.6556 val_loss=0.5205 (improved) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 10/30: train_loss=0.6265 val_loss=1.4079 (no improve 1/5) — 13.0s
    [cnn_aug_12.2_s42_f10] epoch 11/30: train_loss=0.60

## 5. Embeddings + representation shift

In [10]:
from sklearn.metrics.pairwise import rbf_kernel
from scipy.stats import wasserstein_distance

def mmd2(X,Y,g=None):
    if g is None: g=1.0/X.shape[1]
    return float(rbf_kernel(X,X,g).mean()+rbf_kernel(Y,Y,g).mean()-2*rbf_kernel(X,Y,g).mean())

if RUN_EMBED and (RESULTS/"regimes.json").exists():
    torch.manual_seed(SEED)
    reg=json.load(open(RESULTS/"regimes.json")); classes=reg["resnet50"]["clean_to_clean"]["classes"]; nc=len(classes)
    model=MODELS["resnet50"](nc).to(DEVICE)
    ds_tr,_=load_split(US8K_ROWS,None,list(range(1,10)),"us8k")
    ds_te,_=load_split(US8K_ROWS,None,[10],"us8k")
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=LR_RESNET)
    crit=nn.CrossEntropyLoss()
    tl=DataLoader(ds_tr,batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
    vl=DataLoader(torch.utils.data.Subset(ds_tr,list(range(min(500,len(ds_tr))))),
                  batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
    best=float("inf"); pat=0; bst=None
    for ep in range(EPOCHS):
        model.train()
        for X,y in tl:
            X,y=X.to(DEVICE),y.to(DEVICE); opt.zero_grad(); crit(model(X),y).backward(); opt.step()
        model.eval(); vl_=sum(crit(model(X.to(DEVICE)),y.to(DEVICE)).item() for X,y in vl)/len(vl)
        if vl_<best: best=vl_; pat=0; bst={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=PATIENCE: break
    model.load_state_dict(bst); model.eval()
    def embed(ds):
        out=[]
        with torch.no_grad():
            for X,_ in DataLoader(ds,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True):
                out.append(model.get_embedding(X.to(DEVICE)).cpu().numpy())
        return np.concatenate(out)
    Ec=embed(ds_te); shift={}
    for k in KEY_MODES:
        ds_d,_=load_split(US8K_ROWS,k,[10],"us8k"); Ed=embed(ds_d); n=min(len(Ec),len(Ed))
        shift[k]=dict(mmd2=mmd2(Ec,Ed),
                      wasserstein=float(np.mean([wasserstein_distance(Ec[:n,j],Ed[:n,j]) for j in range(Ec.shape[1])])))
        print(f"clean vs AMR{k}: MMD^2={shift[k]['mmd2']:.4f} Wass={shift[k]['wasserstein']:.4f}")
    json.dump(shift,open(RESULTS/"representation_shift.json","w"),indent=2)
    del model,bst; torch.cuda.empty_cache()
    try:
        import umap, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        allX=np.concatenate([Ec]+[embed(load_split(US8K_ROWS,k,[10],"us8k")[0]) for k in KEY_MODES])
        lab=["clean"]*len(Ec)+sum([[f"amr{k}"]*len(Ec) for k in KEY_MODES],[])
        U=umap.UMAP(random_state=SEED).fit_transform(allX)
        fig,ax=plt.subplots(figsize=(5,4))
        for L,col in zip(["clean"]+[f"amr{k}" for k in KEY_MODES],["#4477AA","#EE6677","#CCBB44"]):
            mk=[x==L for x in lab]; ax.scatter(U[mk,0],U[mk,1],s=6,alpha=.5,label=L,color=col)
        ax.legend(); ax.set_title("Penultimate embeddings: clean vs AMR-NB"); plt.tight_layout()
        plt.savefig(FIGS/"umap_repr_shift.png",dpi=140); plt.close(); print("UMAP saved.")
    except Exception as e: print("UMAP skipped:",e)

import shutil; shutil.make_archive("results_file2","zip","results")
print("\nDone. Download results_file2.zip then run File 3.")
print("Files:",sorted(p.name for p in RESULTS.glob("*")))

clean vs AMR4.75: MMD^2=0.0736 Wass=0.7729
clean vs AMR12.2: MMD^2=0.0440 Wass=0.6229
UMAP skipped: name 'model' is not defined

Done. Download results_file2.zip then run File 3.
Files: ['acoustic_profile.json', 'bitrate_sweep.json', 'ckpt', 'figs', 'hf_fraction_per_class.json', 'mismatch_env_vs_speech.json', 'regimes.json', 'representation_shift.json']
